<a href="https://colab.research.google.com/github/kawastony/grok-notes-version-2/blob/main/Cosmology_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install cobaya getdist camb

from pathlib import Path
Path("/content/chains").mkdir(exist_ok=True)
Path("/content/packages").mkdir(exist_ok=True)

!cobaya-install camb \
  planck_2018_lowl.TT planck_2018_lowl.EE \
  planck_2018_highl_plik.TTTEEE_lite_native planck_2018_lensing.native \
  -p /content/packages

!git clone --depth 1 https://github.com/CobayaSampler/bao_data.git /content/bao_data

import camb
print("CAMB", camb.__version__)

In [ ]:
import numpy as np
from pathlib import Path
import glob

PHI = (1 + np.sqrt(5)) / 2
W0 = float(-PHI / 2)   # ≈ -0.8090169944
WA = float(-1 / PHI)   # ≈ -0.6180339887
PACKAGES = "/content/packages"
CHAINS = "/content/chains"
BAO = Path("/content/bao_data")

print("Frozen-φ: w0 =", W0, " wa =", WA)

candidates = sorted(BAO.rglob("*mean*"))
print("BAO mean-like files:")
for p in candidates[:30]:
    print(" ", p)

MEAN = COV = None
for m in candidates:
    for tag in ("cov", "Cov", "COV"):
        c = Path(str(m).replace("mean", tag).replace("Mean", tag))
        if c.exists():
            MEAN, COV = str(m), str(c)
            break
    if MEAN:
        break

print("MEAN:", MEAN)
print("COV:", COV)
# If still None, open /content/bao_data and set paths manually.

In [ ]:
from pathlib import Path

def write(path, text):
    Path(path).write_text(text.strip() + "\n")
    print("Wrote", path)

def params_lcdm():
    return """
  logA:
    prior: {min: 2.5, max: 3.5}
    ref: {dist: norm, loc: 3.044, scale: 0.014}
    proposal: 0.001
    latex: \\log(10^{10} A_s)
    drop: true
  As:
    value: "lambda logA: 1e-10*np.exp(logA)"
    latex: A_s
  ns:
    prior: {min: 0.9, max: 1.1}
    ref: {dist: norm, loc: 0.965, scale: 0.004}
    proposal: 0.002
    latex: n_s
  theta_MC_100:
    prior: {min: 1.0, max: 1.1}
    ref: {dist: norm, loc: 1.041, scale: 0.0004}
    proposal: 0.0002
    latex: 100\\theta_{MC}
    drop: true
  cosmomc_theta:
    value: "lambda theta_MC_100: 1.e-2*theta_MC_100"
    derived: false
  ombh2:
    prior: {min: 0.02, max: 0.025}
    ref: {dist: norm, loc: 0.0224, scale: 0.0001}
    proposal: 0.0001
    latex: \\Omega_b h^2
  omch2:
    prior: {min: 0.08, max: 0.16}
    ref: {dist: norm, loc: 0.12, scale: 0.001}
    proposal: 0.0005
    latex: \\Omega_c h^2
  tau:
    prior: {min: 0.01, max: 0.2}
    ref: {dist: norm, loc: 0.055, scale: 0.006}
    proposal: 0.003
    latex: \\tau_{reio}
  H0:
    latex: H_0
  omegam:
    latex: \\Omega_m
  sigma8:
    latex: \\sigma_8
  A_planck:
    prior: {min: 0.9, max: 1.1}
    ref: {dist: norm, loc: 1.0, scale: 0.0025}
    proposal: 0.0005
    latex: A_{Planck}
"""

def params_phi(w0, wa):
    return params_lcdm() + f"""
  w:
    value: {w0}
    latex: w_0
  wa:
    value: {wa}
    latex: w_a
"""

PL = """
  planck_2018_lowl.TT: null
  planck_2018_lowl.EE: null
  planck_2018_highl_plik.TTTEEE_lite_native: null
  planck_2018_lensing.native: null
"""

def desi_block(mean, cov):
    return f"""
  bao.generic:
    measurements_file: {mean}
    cov_file: {cov}
"""

def make(name, likes, params, use_ppf=False):
    de = "\n      dark_energy_model: DarkEnergyPPF" if use_ppf else ""
    text = f"""
theory:
  camb:
    extra_args:
      lens_potential_accuracy: 1
      num_massive_neutrinos: 1
      nnu: 3.044{de}

likelihood:
{likes}

params:
{params}

sampler:
  mcmc:
    Rminus1_stop: 0.05
    Rminus1_cl_stop: 0.2
    max_tries: 10000
    burn_in: 20

output: {CHAINS}/{name}
packages_path: {PACKAGES}
"""
    write(f"/content/{name}.yaml", text)

# Planck only
make("stage1_lcdm_planck", PL, params_lcdm(), use_ppf=False)
make("stage1_phi_planck", PL, params_phi(W0, WA), use_ppf=True)

# DESI only + joint (if files found)
if MEAN and COV:
    D = desi_block(MEAN, COV)
    make("stage0_lcdm_desi", D, params_lcdm(), use_ppf=False)
    make("stage0_phi_desi", D, params_phi(W0, WA), use_ppf=True)
    make("stage2_lcdm_planck_desi", PL + D, params_lcdm(), use_ppf=False)
    make("stage2_phi_planck_desi", PL + D, params_phi(W0, WA), use_ppf=True)
else:
    print("DESI paths missing — set MEAN/COV then re-run this cell")

print("Done. Frozen point:", W0, WA)
!ls -la /content/*.yaml

In [ ]:
!cobaya-run /content/stage0_lcdm_desi.yaml -f

In [ ]:
!cobaya-run /content/stage0_phi_desi.yaml -f

In [ ]:
!cobaya-run /content/stage1_lcdm_planck.yaml -f

In [ ]:
!cobaya-run /content/stage1_phi_planck.yaml -f

In [ ]:
!cobaya-run /content/stage2_lcdm_planck_desi.yaml -f

In [ ]:
!cobaya-run /content/stage2_phi_planck_desi.yaml -f

In [ ]:
!cp -r /content/chains /content/drive/MyDrive/cosmo_chains_backup

In [ ]:
from getdist import loadMCSamples, plots
import glob, os

CHAINS = "/content/chains"

def root_for(hint):
    txts = glob.glob(f"{CHAINS}/*{hint}*.txt")
    if not txts:
        print("No txt for", hint)
        return None
    # cobaya roots look like .../name.1.txt → file_root = .../name
    roots = sorted(set(t.rsplit(".", 2)[0] for t in txts))
    print(hint, "→", roots)
    return roots[0]

# Main comparison = Stage 2 joint
r_l = root_for("stage2_lcdm")
r_p = root_for("stage2_phi")

# Or Stage 1 only:
# r_l = root_for("stage1_lcdm")
# r_p = root_for("stage1_phi")

s_l = loadMCSamples(r_l, settings={"ignore_rows": 0.3})
s_p = loadMCSamples(r_p, settings={"ignore_rows": 0.3})

for name, s in [("ΛCDM", s_l), ("frozen-φ", s_p)]:
    print(f"\n=== {name} ===")
    for p in ["H0", "omegam", "sigma8", "ns"]:
        try:
            print(" ", s.getInlineLatex(p, limit=1))
        except Exception as e:
            print(" ", p, e)

g = plots.get_subplot_plotter()
g.triangle_plot([s_l, s_p], ["H0", "omegam", "sigma8", "ns"],
                filled=True, legend_labels=["ΛCDM", "frozen-φ"])